## Датасет собран из базы данных переписи 1994 года и содержит данные о доходах.
### Информация о данных:
* age: continuous.
* workclass: Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked.
* fnlwgt: continuous.
* education: Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc, 9th, 7th-8th, 12th, * Masters, 1st-4th, 10th, Doctorate, 5th-6th, Preschool.
* education-num: continuous.
* marital-status: Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse.
* occupation: Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces.
* relationship: Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried.
* race: White, Asian-Pac-Islander, Amer-Indian-Eskimo, Other, Black.
* sex: Female, Male.
* capital-gain: continuous.
* capital-loss: continuous.
* hours-per-week: continuous.
* native-country: United-States, Cambodia, England, Puerto-Rico, Canada, Germany, Outlying-US(Guam-USVI-etc), India, Japan, Greece, South, China, Cuba, Iran, Honduras, Philippines, Italy, Poland, Jamaica, Vietnam, Mexico, Portugal, Ireland, France, Dominican-Republic, Laos, Ecuador, Taiwan, Haiti, Columbia, Hungary, Guatemala, Nicaragua, Scotland, Thailand, Yugoslavia, El-Salvador, Trinadad&Tobago, Peru, Hong, Holand-Netherlands.
* salary: >50K,<=50K

## Проведите анализ данных при помощи Pandas выполнив поставленные задачи.
#### 

In [1]:
import pandas as pd


In [2]:
# загружаем датасет
data = pd.read_csv("./data/adult.data.csv")
data

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


**1. Посчитайте, сколько мужчин и женщин (признак *sex*) представлено в этом датасете**

In [3]:
m_f_count = data.groupby("sex", as_index=False).count()
m_f_count['count'] = m_f_count['age']
m_f_count = m_f_count[['sex', 'count']]
m_f_count


,sex,count
0,Female,10771
1,Male,21790


**2. Каков средний возраст мужчин (признак *age*) по всему датасету?**

In [4]:
avg_age_male = data.query('sex == "Male"')['age'].mean().round(1)
print(avg_age_male)

39.4


**3. Какова доля граждан Соединенных Штатов (признак *native-country*)?**

In [5]:
native_USA = round(data.query('`native-country` == "United-States"').shape[0] / data.shape[0] , 3)
native_USA

0.896

**4-5. Рассчитайте среднее значение и среднеквадратичное отклонение возраста тех, кто получает более 50K в год (признак *salary*) и тех, кто получает менее 50K в год**

In [6]:
salary = data[['age','salary']]
age_std = salary.groupby('salary', as_index=False).std().round(2)
age_std 

,salary,age
0,<=50K,14.02
1,>50K,10.52


**6. Правда ли, что люди, которые получают больше 50k, имеют минимум высшее образование? (признак *education – Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters* или *Doctorate*)**

In [7]:
h_edu= ['Bachelors', 'Prof-school', 'Assoc-acdm', 'Assoc-voc', 'Masters', 'Doctorate']
data['education_level'] = data['education'].apply(
    lambda x: 'High' if x in h_edu else 'Low'
)
T1 = data.groupby(['salary', 'education_level']).size().reset_index(name='count')
if not T1.query('salary == ">50K" and education_level == "Low"').empty:
    print("Предположение отклонено: среди получающих >50K есть люди без высшего образования.")
else:
    print("Предположение подтверждено: все получающие >50K имеют высшее образование.")

Предположение отклонено: среди получающих >50K есть люди без высшего образования.


In [8]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(data['education_level'], data['salary'])
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"χ² = {chi2:.2f}, p-value = {p:.4f}")

if  p<0.05:
    print("Люди с высшим образованием действительно чаще зарабатывают больше 50K")
else:
    print("Наличие высшего образование не влияет на уровень зарплаты")

χ² = 3079.67, p-value = 0.0000
Люди с высшим образованием действительно чаще зарабатывают больше 50K


**7. Выведите статистику возраста для каждой расы (признак *race*) и каждого пола. Используйте *groupby* и *describe*. Найдите таким образом максимальный возраст мужчин расы *Asian-Pac-Islander*.**

In [9]:
age_stat = data.groupby(['sex', 'race'])['age'].describe().reset_index()
age_max = age_stat.query('sex == "Male" and race == "Asian-Pac-Islander"')
print(f'Максимальный возраст мужчин расы Asian-Pac-Islander {age_max["max"].values[0].astype(int)}')

Максимальный возраст мужчин расы Asian-Pac-Islander 90


**8. Среди кого больше доля зарабатывающих много (>50K): среди женатых или холостых мужчин (признак *marital-status*)? Женатыми считаем тех, у кого *marital-status* начинается с *Married* (Married-civ-spouse, Married-spouse-absent или Married-AF-spouse), остальных считаем холостыми.**

In [90]:

married= ['Married-civ-spouse', 'Married-spouse-absent' , 'Married-AF-spouse']
data['married_stat'] = data['marital-status'].apply(lambda x: 'married' if x in married else 'non_married')
T2 = data.query('salary == ">50K"').groupby('married_stat').size().reset_index(name='count')
T2['share'] = (T2['count'] / T2['count'].sum()).round(3)

if T2.query('married_stat == "married"')['share'].values[0] > T2.query('married_stat == "non_married"')['share'].values[0]:
    print("Среди женатых доля зарабатывающих много (>50K) больше")
else:
    print("Среди неженатых доля зарабатывающих много (>50K) больше")
T2



Среди женатых доля зарабатывающих много (>50K) больше


,married_stat,count,share
0,married,6736,0.859
1,non_married,1105,0.141


**9. Какое максимальное число часов человек работает в неделю (признак *hours-per-week*)? Сколько людей работают такое количество часов и каков среди них процент зарабатывающих много?**

In [36]:
work_hours_max=data['hours-per-week'].max()
work_hours=data.query('`hours-per-week` == @work_hours_max').shape[0]
print( f'Максимальное число рабочих часов в неделю {work_hours_max}, по нашим данным, столько часов в неделю работают {work_hours} человек.'  )

Максимальное число рабочих часов в неделю 99, по нашим данным, столько часов в неделю работают 85 человек.


**10. Посчитайте среднее время работы (*hours-per-week*) зарабатывающих мало и много (*salary*) для каждой страны (*native-country*).**

In [49]:
avg_whours = data.groupby(['native-country', 'salary'])['hours-per-week'].mean().round(1).reset_index(name='avg_hours_per_week').replace('?', 'country_not_mentioned')
avg_whours   

,native-country,salary,avg_hours_per_week
0,country_not_mentioned,<=50K,40.2
1,country_not_mentioned,>50K,45.5
2,Cambodia,<=50K,41.4
3,Cambodia,>50K,40.0
4,Canada,<=50K,37.9
...,...,...,...
77,United-States,>50K,45.5
78,Vietnam,<=50K,37.2
79,Vietnam,>50K,39.2
80,Yugoslavia,<=50K,41.6


**11.Сгруппируйте людей по возрастным группам *young*, *adult*, *retiree*, где:**
* *young* соответствует 16-35 лет
* *adult* - 35-70 лет
* *retiree* - 70-100 лет

**Проставьте название соответсвтуещей группы для каждого человека в новой колонке AgeGroup**

In [89]:
data['AgeGroup'] = data['age'].apply(lambda x: 'young' if 16 <= x < 35 else 'adult' if 35 <= x < 70 else 'retiree' if 70 <= x < 100 else 'unknown')
age_group= data.groupby('AgeGroup', as_index=False).size()
age_group

,AgeGroup,size
0,adult,17883
1,retiree,629
2,young,14049


**12-13. Определите количество зарабатывающих >50K в каждой из возрастных групп (колонка AgeGroup), а также выведите название возрастной группы, в которой чаще зарабатывают больше 50К (>50K)**


In [65]:
rich_age = data.query('salary == ">50K"').groupby('AgeGroup').size()
max_group = rich_age.idxmax()
print(f'Возрастная группа, в которой чаще зарабатывают больше 50К: {max_group}')


Возрастная группа, в которой чаще зарабатывают больше 50К: adult


**14. Сгруппируйте людей по типу занятости (колонка occupation) и определите количество людей в каждой группе. После чего напишите функциюю фильтрации filter_func, которая будет возвращать только те группы, в которых средний возраст (колонка age) не больше 40 и в которых все работники отрабатывают более 5 часов в неделю (колонка hours-per-week)**

In [88]:
occupation_stats = data.groupby('occupation', as_index=False).agg({'age': 'mean','hours-per-week': 'min','education': 'count'}).rename(columns={'education': 'count_of_people'})
occupation_stats['age'] = occupation_stats['age'].round(1)

def filter_func(x):
    return x['age'] <= 40 and x['hours-per-week'] > 5
filtered_groups = occupation_stats[occupation_stats.apply(filter_func, axis=1)]    

filtered_groups

,occupation,age,hours-per-week,count_of_people
2,Armed-Forces,30.2,8,9
